In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd
import os

print(os.listdir("/kaggle/input"))

In [ ]:
DATA_PATH = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

print(os.listdir(DATA_PATH))

In [ ]:
import pydicom
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
train_series = pd.read_csv(os.path.join(DATA_PATH, "train_series.csv"))

test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))
test_series = pd.read_csv(os.path.join(DATA_PATH, "test_series.csv"))

print("Train shape:", train.shape)
print("Train series shape:", train_series.shape)
print("Test shape:", test.shape)
print("Test series shape:", test_series.shape)

In [ ]:
display(train.head())

In [ ]:
display(train_series.head())

In [ ]:
print("Train columns:")
print(train.columns.tolist())

print("\nTrain series columns:")
print(train_series.columns.tolist())

In [ ]:
label_columns = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

In [ ]:
print(train[label_columns].head())

In [ ]:
label_counts = train[label_columns].sum()

print(label_counts)

In [ ]:
label_percentages = train[label_columns].mean() * 100

print(label_percentages.round(2))

In [ ]:
print(train_series["Anatomical_Plane"].value_counts())
print(train_series["Fluid_Sensitive"].value_counts())
print(train_series["Fat_Suppression"].value_counts())

In [ ]:
display(
    train_series[
        [
            "Fluid_Sensitive",
            "Fat_Suppression",
            "Anatomical_Plane"
        ]
    ].drop_duplicates()
)


In [ ]:
study_id = train_series["StudyInstanceUID"].iloc[0]

print("Study ID:")
print(study_id)

In [ ]:
study_series = train_series[
    train_series["StudyInstanceUID"] == study_id
]

display(study_series)

In [ ]:
series_id = study_series["SeriesInstanceUID"].iloc[0]

print("Series ID:")
print(series_id)

In [ ]:
series_path = os.path.join(
    DATA_PATH,
    "train_series",
    study_id,
    series_id
)

print(series_path)

In [ ]:
dcm_files = [
    f for f in os.listdir(series_path)
    if f.lower().endswith(".dcm")
]

print("Number of DICOM slices:", len(dcm_files))
print("First few files:", dcm_files[:5])

In [ ]:
dcm_path = os.path.join(
    series_path,
    dcm_files[0]
)

ds = pydicom.dcmread(dcm_path)

print(ds)

In [ ]:
image = ds.pixel_array.astype(np.float32)

print("Image shape:", image.shape)
print("Minimum:", image.min())
print("Maximum:", image.max())

In [ ]:
plt.figure(figsize=(6, 6))

plt.imshow(image, cmap="gray")

plt.title("Raw MRI Slice")
plt.axis("off")

plt.show()

In [ ]:
tags = [
    "PhotometricInterpretation",
    "BitsAllocated",
    "BitsStored",
    "PixelRepresentation",
    "RescaleSlope",
    "RescaleIntercept",
    "ImageOrientationPatient",
    "ImagePositionPatient",
    "SliceThickness",
    "PixelSpacing"
]

for tag in tags:
    if hasattr(ds, tag):
        print(tag, ":", getattr(ds, tag))

In [ ]:
print("Number of DICOM slices:", len(dcm_files))
print("Image shape:", image.shape)
print("Minimum:", image.min())
print("Maximum:", image.max())

In [ ]:
slices = []

for file_name in dcm_files:
    file_path = os.path.join(series_path, file_name)
    
    ds = pydicom.dcmread(file_path)
    
    slices.append(ds)

print("Number of slices loaded:", len(slices))

In [ ]:
for i in range(len(slices)):
    ds = slices[i]
    
    if hasattr(ds, "ImagePositionPatient"):
        print(i, ds.ImagePositionPatient)

In [ ]:
first_ds = slices[0]

print("Image Orientation:")
print(first_ds.ImageOrientationPatient)

print("\nImage Position:")
print(first_ds.ImagePositionPatient)

In [ ]:
orientation = np.array(
    first_ds.ImageOrientationPatient,
    dtype=np.float32
)

row_direction = orientation[:3]
column_direction = orientation[3:]

slice_normal = np.cross(
    row_direction,
    column_direction
)

print("Row direction:", row_direction)
print("Column direction:", column_direction)
print("Slice normal:", slice_normal)

In [ ]:
slice_information = []

for ds in slices:
    
    position = np.array(
        ds.ImagePositionPatient,
        dtype=np.float32
    )
    
    location = np.dot(
        position,
        slice_normal
    )
    
    slice_information.append(
        (location, ds)
    )

print("First few slice locations:")

for item in slice_information[:5]:
    print(item[0])

In [ ]:
slice_information.sort(
    key=lambda x: x[0]
)

In [ ]:
ordered_slices = []

for location, ds in slice_information:
    ordered_slices.append(ds)

print("Ordered slices:", len(ordered_slices))

In [ ]:
volume = []

for ds in ordered_slices:
    
    image = ds.pixel_array.astype(np.float32)
    
    volume.append(image)

volume = np.stack(volume)

print("Volume shape:", volume.shape)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

indices = [0, 5, 10, 15, 21]

for i in range(5):
    
    index = indices[i]
    
    axes[i].imshow(
        volume[index],
        cmap="gray"
    )
    
    axes[i].set_title(
        f"Slice {index}"
    )
    
    axes[i].axis("off")

plt.show()

In [ ]:
def load_series(study_id, series_id):
    
    # Path to the series folder
    series_path = os.path.join(
        DATA_PATH,
        "train_series",
        study_id,
        series_id
    )
    
    # Get all DICOM files
    dcm_files = []
    
    for file_name in os.listdir(series_path):
        if file_name.lower().endswith(".dcm"):
            dcm_files.append(file_name)
    
    # Read all DICOM files
    slices = []
    
    for file_name in dcm_files:
        
        file_path = os.path.join(
            series_path,
            file_name
        )
        
        ds = pydicom.dcmread(file_path)
        slices.append(ds)
    
    # Calculate slice direction
    first_ds = slices[0]
    
    orientation = np.array(
        first_ds.ImageOrientationPatient,
        dtype=np.float32
    )
    
    row_direction = orientation[:3]
    column_direction = orientation[3:]
    
    slice_normal = np.cross(
        row_direction,
        column_direction
    )
    
    # Calculate position of each slice
    slice_information = []
    
    for ds in slices:
        
        position = np.array(
            ds.ImagePositionPatient,
            dtype=np.float32
        )
        
        location = np.dot(
            position,
            slice_normal
        )
        
        slice_information.append(
            (location, ds)
        )
    
    # Sort slices
    slice_information.sort(
        key=lambda x: x[0]
    )
    
    # Create ordered volume
    volume = []
    
    for location, ds in slice_information:
        
        image = ds.pixel_array.astype(np.float32)
        
        volume.append(image)
    
    volume = np.stack(volume)
    
    return volume

In [ ]:
volume = load_series(
    study_id,
    series_id
)

print("Volume shape:", volume.shape)
print("Minimum:", volume.min())
print("Maximum:", volume.max())

In [ ]:
def normalize_image(image):
    
    low = np.percentile(image, 1)
    high = np.percentile(image, 99)
    
    image = np.clip(
        image,
        low,
        high
    )
    
    image = (
        image - low
    ) / (
        high - low + 1e-8
    )
    
    return image

In [ ]:
slice_image = volume[10]

normalized = normalize_image(
    slice_image
)

print("Before normalization:")
print("Min:", slice_image.min())
print("Max:", slice_image.max())

print("\nAfter normalization:")
print("Min:", normalized.min())
print("Max:", normalized.max())

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.imshow(
    slice_image,
    cmap="gray"
)

plt.title("Original")
plt.axis("off")


plt.subplot(1, 2, 2)

plt.imshow(
    normalized,
    cmap="gray"
)

plt.title("Normalized")
plt.axis("off")

plt.show()

In [ ]:
def resize_image(image, size=224):
    
    image_pil = Image.fromarray(
        (image * 255).astype(np.uint8)
    )
    
    image_pil = image_pil.resize(
        (size, size),
        Image.Resampling.BILINEAR
    )
    
    image_resized = np.array(
        image_pil
    ).astype(np.float32) / 255.0
    
    return image_resized

In [ ]:
resized = resize_image(
    normalized
)

print("Original shape:", normalized.shape)
print("Resized shape:", resized.shape)

In [ ]:
image_3ch = np.stack(
    [resized, resized, resized],
    axis=-1
)

print("Shape:", image_3ch.shape)

In [ ]:
plt.figure(figsize=(6, 6))

plt.imshow(image_3ch)

plt.title("Preprocessed MRI Slice")

plt.axis("off")

plt.show()